# Validity — semantic shift vs. stochastic conformance

The claim under test: an activity whose **embedding shifts** between two logs should also show a
**lower stochastic conformance** between the two logs' local behaviour around it.

Per comparison pair `L1` vs `L2`, in this order:

1. **Embedding comparison** — one CWINDOW compass per pair, then a frozen-output model per log,
   so both logs' activity embeddings live in one space (`log_comparison_cwindow`). The
   per-activity cross-log cosine similarity `sim` ranks the shared activities; the bottom decile
   is *shifted*,
   the top decile *stable*.
2. **Projection** — for one selected activity at a time, both logs are cut down to that
   activity's local context `t-c .. t+c`, every occurrence becoming its own case.
3. **EMSC** — the two projected sub-logs are compared as stochastic languages
   (Earth Mover's Stochastic Conformance = 1 - the Wasserstein distance).

A pair is then reduced to **one summary row**, written to disk, and every object it produced is
dropped before the next pair starts — the run never holds more than one pair's data. The final
table is assembled by reading those rows back, so it survives an interrupted run.

## Configuration

`RECOMPUTE` decides whether the pair loop runs at all. With `False` nothing is
trained or projected: the results section simply reads the per-pair rows already on disk.

Settings shared by the whole experiment (`CONTEXT_WINDOW`, `EMBEDDING_DIM`, `UNIT_NORM`,
`BALANCE_COMPASS`, `QUANTILE_SHIFTED`, `QUANTILE_STABLE`) are globals, so every pair reads the
same span of local behaviour into embeddings of the same size. A log named `X` is read from `LOGS_DIR/X.xes`.

`PAIRS` lists the comparisons: one entry per `L1` vs `L2`, each spelling out its own training
schedule (learning rates and epochs for the compass and for the per-log retraining), so a single
pair can be tuned without touching the others. Nothing is inherited — omitting or misspelling
one raises.

In [13]:
# --- Imports -----------------------------------------------------------------
import gc
import os

import numpy as np
import pandas as pd
from IPython.display import display
from tqdm.auto import tqdm

import pm4py
import emsc_runner
import log_comparison_cwindow as lcw

# --- Reproducibility ---------------------------------------------------------
SEED = 42
lcw.set_seed(SEED)

# --- Fixed settings ----------------------------------------------------------
DEVICE = "cpu"  # small full-batch runs; MPS is slower for these
EMSC_TIMEOUT = None  # seconds allowed per EMSC; the activity is skipped if it overruns
#                         (None = wait forever)
LOGS_DIR = "logs"  # a log named X is read from LOGS_DIR/X.xes
RESULTS_DIR = "results/01_validity"  # one CSV per pair, one row each

# False = do not train or project anything; just read the rows already in RESULTS_DIR
RECOMPUTE = False

# --- Global experiment settings (identical for every pair) -------------------
CONTEXT_WINDOW = 4  # context window c; the projection spans t-c .. t+c
EMBEDDING_DIM = 32  # d, the activity embedding dimension
UNIT_NORM = True  # keep the activity embeddings on the unit sphere: each context
#                         embedding is L2-normalised before it enters the concatenated context
#                         vector, and the read-out is normalised too. The model cannot spend
#                         capacity on embedding length, so an activity is described by the
#                         direction alone -- which is exactly what the cosine reads
BALANCE_COMPASS = (
    True  # scale the smaller log's case counts by the size imbalance so both
)
#                         logs weigh equally in the compass loss; nothing is resampled, and
#                         the per-log retraining stays unweighted
QUANTILE_SHIFTED = 0.10  # cosine <= this quantile -> shifted
QUANTILE_STABLE = 0.90  # cosine >= this quantile -> stable

# --- One comparison entry ----------------------------------------------------
# Every pair states its own full set of per-pair hyperparameters, so any pair can be tuned
# without touching the others. Nothing is inherited: a missing or misspelled hyperparameter
# raises instead of silently falling back to a default.
#
#   compass_learning_rate / compass_epochs    compass stage (the shared space)
#   log_learning_rate / log_epochs            per-log frozen-output retraining
HYPERPARAM_KEYS = (
    "compass_learning_rate",
    "compass_epochs",
    "log_learning_rate",
    "log_epochs",
)


def pair(l1, l2, **hyperparams):
    """One comparison: the two log names plus every hyperparameter it runs with."""
    missing = [k for k in HYPERPARAM_KEYS if k not in hyperparams]
    unknown = [k for k in hyperparams if k not in HYPERPARAM_KEYS]
    if missing or unknown:
        raise ValueError(
            f"{l1} vs {l2}: "
            + (f"missing hyperparameters {missing}. " if missing else "")
            + (f"unknown hyperparameters {unknown}." if unknown else "")
        )
    return {"l1": l1, "l2": l2, **hyperparams}


def pair_id(p):
    return f"{p['l1']} vs {p['l2']}"


def pair_path(p):
    """The file this pair's single result row is written to."""
    return f"{RESULTS_DIR}/{p['l1']}__{p['l2']}.csv"


# --- The experiment ----------------------------------------------------------
PAIRS = [
    # --- BPIC11 age ---
    pair(
        "BPIC11_age_low",
        "BPIC11_age_high",
        compass_learning_rate=1e-2,
        compass_epochs=1500,
        log_learning_rate=1e-2,
        log_epochs=500,
    ),
    # --- BPIC15 municipalities ---
    pair(
        "BPIC15_M1",
        "BPIC15_M2",
        compass_learning_rate=1e-2,
        compass_epochs=1500,
        log_learning_rate=1e-2,
        log_epochs=500,
    ),
    pair(
        "BPIC15_M1",
        "BPIC15_M3",
        compass_learning_rate=1e-2,
        compass_epochs=1500,
        log_learning_rate=1e-2,
        log_epochs=500,
    ),
    pair(
        "BPIC15_M1",
        "BPIC15_M4",
        compass_learning_rate=1e-2,
        compass_epochs=1500,
        log_learning_rate=1e-2,
        log_epochs=500,
    ),
    pair(
        "BPIC15_M1",
        "BPIC15_M5",
        compass_learning_rate=1e-2,
        compass_epochs=1500,
        log_learning_rate=1e-2,
        log_epochs=500,
    ),
    pair(
        "BPIC15_M2",
        "BPIC15_M3",
        compass_learning_rate=1e-2,
        compass_epochs=1500,
        log_learning_rate=1e-2,
        log_epochs=500,
    ),
    pair(
        "BPIC15_M2",
        "BPIC15_M4",
        compass_learning_rate=1e-2,
        compass_epochs=1500,
        log_learning_rate=1e-2,
        log_epochs=500,
    ),
    pair(
        "BPIC15_M2",
        "BPIC15_M5",
        compass_learning_rate=1e-2,
        compass_epochs=1500,
        log_learning_rate=1e-2,
        log_epochs=500,
    ),
    pair(
        "BPIC15_M3",
        "BPIC15_M4",
        compass_learning_rate=1e-2,
        compass_epochs=1500,
        log_learning_rate=1e-2,
        log_epochs=500,
    ),
    pair(
        "BPIC15_M3",
        "BPIC15_M5",
        compass_learning_rate=1e-2,
        compass_epochs=1500,
        log_learning_rate=1e-2,
        log_epochs=500,
    ),
    pair(
        "BPIC15_M4",
        "BPIC15_M5",
        compass_learning_rate=1e-2,
        compass_epochs=1500,
        log_learning_rate=1e-2,
        log_epochs=500,
    ),
    # --- BPIC18 departments ---
    pair(
        "BPIC18_D4e",
        "BPIC18_D6b",
        compass_learning_rate=1e-2,
        compass_epochs=1000,
        log_learning_rate=1e-2,
        log_epochs=500,
    ),
    pair(
        "BPIC18_D4e",
        "BPIC18_Dd4",
        compass_learning_rate=1e-2,
        compass_epochs=1000,
        log_learning_rate=1e-2,
        log_epochs=500,
    ),
    pair(
        "BPIC18_D4e",
        "BPIC18_De7",
        compass_learning_rate=1e-2,
        compass_epochs=1000,
        log_learning_rate=1e-2,
        log_epochs=500,
    ),
    pair(
        "BPIC18_D6b",
        "BPIC18_Dd4",
        compass_learning_rate=1e-2,
        compass_epochs=1000,
        log_learning_rate=1e-2,
        log_epochs=500,
    ),
    pair(
        "BPIC18_D6b",
        "BPIC18_De7",
        compass_learning_rate=1e-2,
        compass_epochs=1000,
        log_learning_rate=1e-2,
        log_epochs=500,
    ),
    pair(
        "BPIC18_Dd4",
        "BPIC18_De7",
        compass_learning_rate=1e-2,
        compass_epochs=1000,
        log_learning_rate=1e-2,
        log_epochs=500,
    ),
]

os.makedirs(RESULTS_DIR, exist_ok=True)
print(f"{len(PAIRS)} pairs: {', '.join(pair_id(p) for p in PAIRS)}")

17 pairs: BPIC11_age_low vs BPIC11_age_high, BPIC15_M1 vs BPIC15_M2, BPIC15_M1 vs BPIC15_M3, BPIC15_M1 vs BPIC15_M4, BPIC15_M1 vs BPIC15_M5, BPIC15_M2 vs BPIC15_M3, BPIC15_M2 vs BPIC15_M4, BPIC15_M2 vs BPIC15_M5, BPIC15_M3 vs BPIC15_M4, BPIC15_M3 vs BPIC15_M5, BPIC15_M4 vs BPIC15_M5, BPIC18_D4e vs BPIC18_D6b, BPIC18_D4e vs BPIC18_Dd4, BPIC18_D4e vs BPIC18_De7, BPIC18_D6b vs BPIC18_Dd4, BPIC18_D6b vs BPIC18_De7, BPIC18_Dd4 vs BPIC18_De7


## Helpers

`load_log` reads one log; `project_log_on_context` cuts a log down to one activity's local
context. Both work on the three columns the rest of the notebook needs, and nothing is cached —
a log is read again for every pair it takes part in, so no pair's data outlives its iteration.

In [14]:
CASE_COL, ACTIVITY_COL, TIME_COL = "case:concept:name", "concept:name", "time:timestamp"


def load_log(name):
    """Read the log named ``name`` from ``LOGS_DIR``, reduced to the three needed columns.

    ``concept:name`` is the activity as ``00_preprocessing`` left it -- for BPIC15 that is the
    readable ``activityNameEN`` label, not the internal activity code, which splits one activity
    into several numbered variants. Nothing is relabelled here.
    """
    return pm4py.read_xes(f"{LOGS_DIR}/{name}.xes")[[CASE_COL, ACTIVITY_COL, TIME_COL]]


def project_log_on_context(log_df, activity, c):
    """Project a log onto the local context window around every occurrence of one activity.

    For an occurrence at position ``t`` in a case, the extracted sub-trace spans
    ``t-c, ..., t, ..., t+c``: up to c events of left context (what predicts the activity), the
    activity itself, and c events of right context (what it predicts). Windows are clipped at
    the trace boundaries, and each window becomes its own case ``f"{case_id}_occ{i}"``.

    :param log_df: the event log.
    :param activity: the anchor activity.
    :param c: the context window size.
    :returns: the projected sub-log, with the same columns; empty if the activity never occurs.
    """
    cases = log_df.loc[log_df[ACTIVITY_COL] == activity, CASE_COL].unique()
    log_df = log_df[log_df[CASE_COL].isin(cases)]

    frames = []
    for case_id, case_df in log_df.groupby(CASE_COL, sort=False):
        case_df = case_df.sort_values(TIME_COL, kind="stable").reset_index(drop=True)
        for occ, pos in enumerate(
            np.flatnonzero(case_df[ACTIVITY_COL].values == activity)
        ):
            window = case_df.iloc[max(0, pos - c) : pos + c + 1].copy()
            window[CASE_COL] = f"{case_id}_occ{occ}"
            frames.append(window)

    return pd.concat(frames, ignore_index=True) if frames else log_df.iloc[:0]


# the two quantile columns are named after the settings they follow
COL_Q_SHIFTED = f"sim q{round(QUANTILE_SHIFTED * 100)} (shifted <=)"
COL_Q_STABLE = f"sim q{round(QUANTILE_STABLE * 100)} (stable >=)"

## The run — one pair at a time

For each pair: load the two logs, compare their embeddings, then walk the selected activities —
project both logs onto the activity, run the EMSC on the two projections, keep the number. The
pair is then reduced to one row, written to `RESULTS_DIR`, and everything it allocated is freed.

`emsc_runner.emsc_with_timeout` runs the Ebi call in a throwaway child process and kills it after
`EMSC_TIMEOUT` seconds, so a stuck activity costs at most that and only that activity is lost.
The child process is unavoidable: Ebi is a compiled extension, so a signal-based timeout would
only fire once the call had already finished.

In [15]:
def run_pair(p):
    """Run one comparison end to end and return its single result row."""
    l1, l2 = p["l1"], p["l2"]
    log_1, log_2 = load_log(l1), load_log(l2)

    # --- 1. embedding comparison: one compass, then a frozen-output model per log ---
    model_1, model_2, _ = lcw.compare_cwindow_event_logs(
        log_1,
        log_2,
        names=(l1, l2),
        c=CONTEXT_WINDOW,
        embedding_dim=EMBEDDING_DIM,
        compass_learning_rate=p["compass_learning_rate"],
        compass_epochs=p["compass_epochs"],
        log_learning_rate=p["log_learning_rate"],
        log_epochs=p["log_epochs"],
        balance_compass=BALANCE_COMPASS,
        unit_norm=UNIT_NORM,
        device=DEVICE,
        verbose=False,
    )

    sim = lcw.activity_similarity(model_1, model_2)
    q_shifted = float(np.quantile(sim.values, QUANTILE_SHIFTED))
    q_stable = float(np.quantile(sim.values, QUANTILE_STABLE))
    selected = [("stable", a) for a in sim.index[sim.values >= q_stable]]
    selected += [("shifted", a) for a in sim.index[sim.values <= q_shifted]]

    # --- 2. + 3. projection and EMSC, one selected activity at a time ---
    values = {"stable": [], "shifted": []}
    n_failed = 0
    for group, activity in tqdm(selected, desc=pair_id(p), leave=False):
        sub_1 = project_log_on_context(log_1, activity, CONTEXT_WINDOW)
        sub_2 = project_log_on_context(log_2, activity, CONTEXT_WINDOW)
        if sub_1.empty or sub_2.empty:
            n_failed += 1
            continue
        try:
            values[group].append(
                emsc_runner.emsc_with_timeout(sub_1, sub_2, timeout=EMSC_TIMEOUT)
            )
        except Exception:  # a timeout or an Ebi failure costs this activity only
            n_failed += 1
        del sub_1, sub_2

    stable = np.array(values["stable"])
    shifted = np.array(values["shifted"])
    total = stable.size * shifted.size
    wins = int((stable[:, None] > shifted[None, :]).sum()) if total else 0

    acts_1, acts_2 = (
        set(log_1[ACTIVITY_COL].unique()),
        set(log_2[ACTIVITY_COL].unique()),
    )
    row = {
        "L1": l1,
        "L2": l2,
        "shared acts": len(acts_1 & acts_2),
        "total acts": len(acts_1 | acts_2),
        "sim min": float(sim.values.min()),
        COL_Q_SHIFTED: q_shifted,
        "sim median": float(np.median(sim.values)),
        COL_Q_STABLE: q_stable,
        "sim max": float(sim.values.max()),
        "# shifted": shifted.size,
        "# stable": stable.size,
        "EMSC shifted": shifted.mean() if shifted.size else np.nan,
        "EMSC stable": stable.mean() if stable.size else np.nan,
        "EMSC delta": (stable.mean() - shifted.mean())
        if (stable.size and shifted.size)
        else np.nan,
        "stable > shifted": f"{wins}/{total}",
        "win rate %": 100 * wins / total if total else np.nan,
        "# failed": n_failed,
    }

    del log_1, log_2, model_1, model_2, sim
    return row


if RECOMPUTE:
    for p in tqdm(PAIRS, desc="Comparisons"):
        row = run_pair(p)
        pd.DataFrame([row]).to_csv(
            pair_path(p), index=False
        )  # saved before the next pair
        print(
            f"{pair_id(p)}: EMSC stable {row['EMSC stable']:.3f} vs shifted "
            f"{row['EMSC shifted']:.3f}, win rate {row['win rate %']:.1f}% "
            f"({row['# failed']} failed) -> {pair_path(p)}"
        )
        del row
        gc.collect()
else:
    print(f"RECOMPUTE = False — reading the rows already in {RESULTS_DIR}/")

RECOMPUTE = False — reading the rows already in results/01_validity/


## Results

The single-row files are read back and stacked, one row per comparison pair.

* **shared / total acts** — activity alphabet overlap of the two logs.
* **sim min / q10 / median / q90 / max** — cross-log cosine similarity over the shared
  activities, the set the quantiles are taken over. `q10` is the *shifted* cut-off (cosine at or below it), `q90` the *stable* one.
* **EMSC shifted / stable / delta** — mean EMSC per group and their difference; a positive delta
  is the expected direction.
* **win rate** — share of the (stable, shifted) activity comparisons *within this pair* where the
  stable activity reaches the higher EMSC. 50% = no relationship, > 50% supports the hypothesis.

In [16]:
paths = [pair_path(p) for p in PAIRS if os.path.exists(pair_path(p))]
missing = [pair_id(p) for p in PAIRS if not os.path.exists(pair_path(p))]
if missing:
    print(f"no result row for {missing} — set RECOMPUTE = True and rerun")
if not paths:
    raise RuntimeError(f"{RESULTS_DIR}/ holds no result rows")

results = pd.concat((pd.read_csv(path) for path in paths), ignore_index=True)
results.to_csv("results/01_validity_summary.csv", index=False)

display(
    results.style.format(
        {
            "sim min": "{:.3f}",
            COL_Q_SHIFTED: "{:.3f}",
            "sim median": "{:.3f}",
            COL_Q_STABLE: "{:.3f}",
            "sim max": "{:.3f}",
            "EMSC shifted": "{:.3f}",
            "EMSC stable": "{:.3f}",
            "EMSC delta": "{:+.3f}",
            "win rate %": "{:.1f}",
        }
    )
    .background_gradient(subset=["win rate %"], cmap="RdYlGn", vmin=0, vmax=100)
    .hide(axis="index")
)

L1,L2,shared acts,total acts,sim min,sim q10 (shifted <=),sim median,sim q90 (stable >=),sim max,# shifted,# stable,EMSC shifted,EMSC stable,EMSC delta,stable > shifted,win rate %,# failed
BPIC11_age_low,BPIC11_age_high,367,624,-0.107,0.380,0.674,0.953,0.993,37,37,0.253,0.772,+0.519,1368/1369,99.9,0
BPIC15_M1,BPIC15_M2,262,331,0.190,0.468,0.779,0.949,0.981,27,27,0.267,0.582,+0.315,698/729,95.7,0
BPIC15_M1,BPIC15_M3,250,316,0.123,0.500,0.794,0.950,0.986,25,25,0.281,0.654,+0.373,605/625,96.8,0
BPIC15_M1,BPIC15_M4,256,305,0.011,0.484,0.786,0.928,0.980,26,26,0.305,0.549,+0.244,594/676,87.9,0
BPIC15_M1,BPIC15_M5,253,321,0.217,0.502,0.789,0.936,0.977,26,26,0.318,0.552,+0.234,608/676,89.9,0
BPIC15_M2,BPIC15_M3,256,325,0.106,0.454,0.783,0.943,0.979,26,26,0.291,0.565,+0.275,619/676,91.6,0
BPIC15_M2,BPIC15_M4,258,318,0.008,0.481,0.776,0.940,0.989,26,26,0.255,0.565,+0.310,660/676,97.6,0
BPIC15_M2,BPIC15_M5,252,337,0.157,0.488,0.815,0.950,0.983,26,26,0.284,0.593,+0.309,657/676,97.2,0
BPIC15_M3,BPIC15_M4,257,292,0.164,0.467,0.781,0.936,0.975,26,26,0.260,0.562,+0.302,630/676,93.2,0
BPIC15_M3,BPIC15_M5,254,308,0.163,0.434,0.769,0.938,0.981,26,26,0.263,0.571,+0.308,664/676,98.2,0
